# Pipeline 1 - Extracao balanceada ATUALIZADA 10500

Le o dataset original do arXiv em JSON Lines dentro da pasta `archive` e gera uma amostra balanceada com 15 categorias x 700 artigos = 10500 linhas.

A variavel principal e `AMOSTRA_TOTAL`. Para testar outro tamanho, mude apenas ela, mantendo um valor divisivel por 15.

## 1. Configuracao

In [ ]:
TARGET_CATS = [
    "cs.LG", "hep-ph", "cs.CV", "cs.AI", "hep-th",
    "quant-ph", "gr-qc", "cs.CL", "cond-mat.mtrl-sci", "astro-ph",
    "cond-mat.mes-hall", "math-ph", "math.MP", "cond-mat.str-el", "cond-mat.stat-mech",
]

AMOSTRA_TOTAL = 10500
N_POR_CAT = AMOSTRA_TOTAL // len(TARGET_CATS)

assert AMOSTRA_TOTAL % len(TARGET_CATS) == 0, 'AMOSTRA_TOTAL precisa ser divisivel por 15 para manter balanceamento exato.'
assert len(TARGET_CATS) == 15, 'Esperado 15 categorias.'

CAMINHO = 'arxiv-metadata-oai-snapshot.json'
SAIDA = 'arxiv_amostra_10500_atualizada.json'

print(f'{len(TARGET_CATS)} categorias x {N_POR_CAT} = {AMOSTRA_TOTAL} linhas')

## 2. Montar Drive e localizar dataset original

O arquivo original de aproximadamente 5 GB deve estar em `archive/arxiv-metadata-oai-snapshot.json`. A celula procura em caminhos diretos e faz uma busca rasa se necessario.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE = None  # opcional: coloque aqui a pasta exata do projeto se quiser evitar busca
NOME_DATASET = 'arxiv-metadata-oai-snapshot.json'


def shallow_find(root, filename, max_depth=3):
    root = os.path.abspath(root)
    root_depth = root.rstrip(os.sep).count(os.sep)
    for current, dirs, files in os.walk(root):
        depth = current.rstrip(os.sep).count(os.sep) - root_depth
        if depth >= max_depth:
            dirs[:] = []
        if filename in files:
            return os.path.join(current, filename)
    return None

candidatos = []
if BASE:
    candidatos.extend([
        os.path.join(BASE, 'archive', NOME_DATASET),
        os.path.join(BASE, NOME_DATASET),
    ])

candidatos.extend([
    os.path.join('/content/drive/MyDrive', 'projetoIA-EquipeLoremIpsum', 'archive', NOME_DATASET),
    os.path.join('/content/drive/MyDrive', 'projetoIA-EquipeLoremIpsum', 'source-arxiv', NOME_DATASET),
    os.path.join('/content/drive/MyDrive', 'archive', NOME_DATASET),
    os.path.join('/content/drive/MyDrive', NOME_DATASET),
])

CAMINHO = next((p for p in candidatos if os.path.exists(p)), None)
if CAMINHO is None:
    print('Dataset not found in direct paths. Starting shallow search in MyDrive, max_depth=3...')
    CAMINHO = shallow_find('/content/drive/MyDrive', NOME_DATASET, max_depth=3)

if CAMINHO is None:
    print('Required dataset not found:', NOME_DATASET)
    print('Direct paths tested:')
    for p in candidatos:
        print(' -', p)
    raise FileNotFoundError(NOME_DATASET)

PROJECT_DIR = os.path.dirname(os.path.dirname(CAMINHO)) if os.path.basename(os.path.dirname(CAMINHO)) == 'archive' else os.path.dirname(CAMINHO)
PIPE = os.path.join(PROJECT_DIR, 'pipelines')
os.makedirs(PIPE, exist_ok=True)
SAIDA = os.path.join(PIPE, 'arxiv_amostra_10500_atualizada.json')

print('Input dataset:', CAMINHO)
print('Output file:', SAIDA)
print('Input size GB:', round(os.path.getsize(CAMINHO) / 1e9, 2))

## 3. Filtrar amostra balanceada por streaming

In [ ]:
import json
from collections import defaultdict

target_set = set(TARGET_CATS)
buckets = defaultdict(list)
restantes = set(TARGET_CATS)
lidos = 0

with open(CAMINHO, encoding='utf-8') as f:
    for linha in f:
        if not restantes:
            break
        lidos += 1
        p = json.loads(linha)
        cats = (p.get('categories') or '').split()
        if not cats:
            continue

        escolha = cats[0] if cats[0] in restantes else None
        if escolha is None:
            for c in cats:
                if c in restantes:
                    escolha = c
                    break
        if escolha is None:
            continue

        buckets[escolha].append({
            'id': p.get('id'),
            'title': (p.get('title') or '').strip(),
            'abstract': (p.get('abstract') or '').strip(),
            'categories': p.get('categories'),
            'primary_category': cats[0],
            'assigned_category': escolha,
            'authors': p.get('authors'),
            'update_date': p.get('update_date'),
        })
        if len(buckets[escolha]) >= N_POR_CAT:
            restantes.discard(escolha)

print(f'Rows read from original dataset: {lidos:,}')
print(f'Completed categories: {len(TARGET_CATS) - len(restantes)}/{len(TARGET_CATS)}')
if restantes:
    print('Warning: categories below target:', sorted(restantes))

## 4. Montar DataFrame e validar

In [ ]:
import pandas as pd

linhas = [row for cat in TARGET_CATS for row in buckets[cat]]
df = pd.DataFrame(linhas)

print('Shape:', df.shape)
print('\nRows per category:')
print(df['assigned_category'].value_counts().reindex(TARGET_CATS))
print('\nDuplicated IDs:', df['id'].duplicated().sum())
df.head()

## 5. Salvar JSON Lines

In [ ]:
df.to_json(SAIDA, orient='records', lines=True, force_ascii=False)
print('Saved:', SAIDA, '|', len(df), 'records')

try:
    from google.colab import files
    files.download(SAIDA)
except Exception as e:
    print('Automatic download unavailable:', e)